# Options IV Surface & Monte Carlo Analysis — NVDA
### Powered by [hpsilab.com](https://hpsilab.com) · Quant Finance API & MCP Server

This notebook demonstrates institutional-grade options analytics on NVIDIA (NVDA) using the `hpsilab-mcp` Python SDK.

**What we'll cover:**
1. Implied Volatility (IV) Radar — term structure & skew
2. Monte Carlo price simulation — 10-day distribution
3. Option chain pressure — Max Pain, dealer gamma exposure
4. Full stock analysis — consensus signal
5. *(Pro)* AI direction prediction — regime-aware ensemble

> All free-tier tools run anonymously. No API key required for cells 1–4.

## 0. Install

In [ ]:
!pip install hpsilab-mcp -q

In [ ]:
from hpsilab_mcp import HpsiMcpClient
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Free tier — no API key needed
client = HpsiMcpClient()
SYMBOL = "NVDA"
print(f"Ready. Analyzing {SYMBOL}...")

---
## 1. Implied Volatility Radar

IV Radar returns the full volatility surface: term structure across expirations, put/call skew, and whether IV is elevated relative to historical vol (HV). High IV → options are expensive; low IV → cheap relative to realized moves.

In [ ]:
iv_data = client.get_iv_radar(SYMBOL)

# Pretty-print the raw response
print(json.dumps(iv_data, indent=2))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    r = iv_data['results'][0]

    symbol    = r['symbol']
    expiry_iv = r['iv30_expiry']
    atm_iv    = r['atm_iv']
    iv_rank   = r['iv_rank']
    iv_pct    = r['iv_percentile']
    risk_rev  = r['risk_reversal_25d']
    squeeze_sc = r['squeeze_score']
    regime    = r['regime']
    hv30      = r['hv30']
    call_iv   = r['call_25d_iv']
    put_iv    = r['put_25d_iv']
    near_iv   = r['near_month_iv']
    far_iv    = r['far_month_iv']
    far_exp   = r['far_month_expiry']
    term_str  = r['term_structure']

    # ── Chart 1: IV Term Structure + Skew ───────────────────────────
    fig1 = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f'IV Term Structure ({term_str.upper()})', 'Vol Skew (25Δ)')
    )
    # Term structure line
    fig1.add_trace(go.Scatter(
        x=[expiry_iv, far_exp],
        y=[near_iv*100, far_iv*100],
        mode='lines+markers',
        line=dict(color='#2A78D6', width=2.5),
        marker=dict(size=8),
        name='IV',
        fill='tozeroy', fillcolor='rgba(42,120,214,0.07)',
    ), row=1, col=1)
    # HV30 reference
    fig1.add_hline(y=hv30*100, line_dash='dash', line_color='#F59E0B',
                   annotation_text=f'HV30 {hv30*100:.1f}%',
                   annotation_position='top right', row=1, col=1)
    # Skew bars
    fig1.add_trace(go.Bar(
        x=['Call 25Δ', 'ATM', 'Put 25Δ'],
        y=[call_iv*100, atm_iv*100, put_iv*100],
        marker_color=['#3B82F6', '#8B5CF6', '#EF4444'],
        text=[f'{v*100:.1f}%' for v in [call_iv, atm_iv, put_iv]],
        textposition='outside',
        name='Skew',
    ), row=1, col=2)
    fig1.update_layout(
        title=dict(
            text=f'IV Surface — {symbol}  |  Spot ${r["spot"]}  |  Regime: {regime}',
            font=dict(size=14)
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        height=360, showlegend=False,
        margin=dict(l=40, r=40, t=80, b=20),
    )
    fig1.update_yaxes(title_text='IV (%)', row=1, col=1)
    fig1.update_yaxes(title_text='IV (%)', row=1, col=2)
    fig1.show()

    # ── Chart 2: IV Structure Dashboard (gauges) ─────────────────────
    fig2 = make_subplots(
        rows=1, cols=3,
        subplot_titles=('IV Rank / Percentile', '25-Delta Risk Reversal', 'Squeeze Score'),
        specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]],
    )
    # IV Rank gauge
    fig2.add_trace(go.Indicator(
        mode='gauge+number',
        value=iv_rank,
        title=dict(text='IV Rank', font=dict(size=13)),
        number=dict(suffix='', font=dict(size=22)),
        gauge=dict(
            axis=dict(range=[0, 100], tickwidth=1, tickcolor='#888'),
            bar=dict(color='#2A78D6'),
            bgcolor='white',
            steps=[
                dict(range=[0, 30],  color='#EAF3DE'),
                dict(range=[30, 70], color='#FAEEDA'),
                dict(range=[70, 100], color='#FCEBEB'),
            ],
            threshold=dict(line=dict(color='#E34948', width=2), thickness=0.75, value=iv_rank),
        ),
    ), row=1, col=1)
    # Risk Reversal
    rr_color = '#E34948' if risk_rev < 0 else '#1BAF7A'
    fig2.add_trace(go.Indicator(
        mode='number+delta',
        value=risk_rev,
        title=dict(text='Risk Reversal (25Δ)', font=dict(size=13)),
        number=dict(font=dict(size=28, color=rr_color)),
        delta=dict(reference=0, relative=False,
                   increasing=dict(color='#1BAF7A'), decreasing=dict(color='#E34948')),
    ), row=1, col=2)
    # Squeeze Score
    sq_color = '#1BAF7A' if squeeze_sc > 0 else '#E34948'
    fig2.add_trace(go.Indicator(
        mode='number+delta',
        value=squeeze_sc,
        title=dict(text='Squeeze Score', font=dict(size=13)),
        number=dict(font=dict(size=28, color=sq_color)),
        delta=dict(reference=0, relative=False,
                   increasing=dict(color='#1BAF7A'), decreasing=dict(color='#E34948')),
    ), row=1, col=3)
    fig2.update_layout(
        title=dict(
            text=f'IV Structure — {symbol}  |  ATM IV {atm_iv*100:.1f}%  |  Regime: {regime}  |  Expiry {expiry_iv}',
            font=dict(size=14)
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        height=340,
        margin=dict(l=40, r=40, t=80, b=20),
    )
    fig2.show()

    print(f'IV Rank: {iv_rank:.1f} | IV Percentile: {iv_pct:.1f}')
    print(f'Risk Reversal 25Δ: {risk_rev:.4f} | Squeeze Score: {squeeze_sc:.2f}')
    print(f'Vol Interpretation: {r["vol_interpretation"]}')

except Exception as e:
    print(f'Visualization skipped: {e}')


---
## 2. Monte Carlo Price Simulation (10-day)

Runs thousands of GBM simulations to model the distribution of possible prices 10 trading days out. Useful for:
- Setting realistic profit targets and stop levels
- Sizing positions relative to tail risk
- Understanding the 90% confidence range vs. current price

In [ ]:
mc_data = client.get_monte_carlo(SYMBOL)
print(json.dumps(mc_data, indent=2))

In [ ]:
try:
    import numpy as np
    from scipy.stats import gaussian_kde

    d          = mc_data
    prices_arr = np.array(d['final_prices'])
    mean_price = d['mean_price']
    lower_90   = d['lower_bound']
    upper_90   = d['upper_bound']
    threshold  = d['threshold']
    prob_above = d['prob_above']
    msg        = d.get('msg', '')

    # Histogram bins
    counts, bin_edges = np.histogram(prices_arr, bins=60)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width   = bin_edges[1] - bin_edges[0]

    # KDE
    kde     = gaussian_kde(prices_arr, bw_method=0.3)
    x_range = np.linspace(prices_arr.min(), prices_arr.max(), 400)
    kde_y   = kde(x_range) * len(prices_arr) * bin_width

    fig = go.Figure()

    # Histogram — blue below, red above threshold
    mask_below = bin_centers <= threshold
    fig.add_trace(go.Bar(
        x=bin_centers[mask_below], y=counts[mask_below],
        width=bin_width * 0.9,
        marker_color='rgba(59,130,246,0.45)',
        marker_line_width=0,
        name='Below threshold',
    ))
    fig.add_trace(go.Bar(
        x=bin_centers[~mask_below], y=counts[~mask_below],
        width=bin_width * 0.9,
        marker_color='rgba(239,68,68,0.45)',
        marker_line_width=0,
        name='Above threshold',
    ))

    # KDE curve
    mask_kde = x_range <= threshold
    fig.add_trace(go.Scatter(
        x=x_range[mask_kde], y=kde_y[mask_kde],
        mode='lines', line=dict(color='#2563EB', width=2),
        name='KDE (below)', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=x_range[~mask_kde], y=kde_y[~mask_kde],
        mode='lines', line=dict(color='#DC2626', width=2),
        name='KDE (above)', showlegend=False,
    ))

    # 90% band
    fig.add_vrect(
        x0=lower_90, x1=upper_90,
        fillcolor='rgba(16,185,129,0.08)',
        layer='below', line_width=0,
        annotation_text='90% range',
        annotation_position='top left',
        annotation_font_color='#10B981',
    )

    # Mean line
    fig.add_vline(
        x=mean_price, line_dash='dot', line_color='#2563EB', line_width=1.8,
        annotation_text=f'Mean ${mean_price:.2f}',
        annotation_position='top right',
        annotation_font_color='#2563EB',
    )

    # Threshold line
    fig.add_vline(
        x=threshold, line_dash='dash', line_color='#EF4444', line_width=1.8,
        annotation_text=f'Threshold ${threshold:.2f}',
        annotation_position='top left',
        annotation_font_color='#EF4444',
    )

    fig.update_layout(
        title=dict(
            text=f'Monte Carlo Price Distribution — NVDA  |  {len(prices_arr):,} simulations  |  Prob above threshold: {prob_above*100:.2f}%',
            font=dict(size=14)
        ),
        xaxis_title='Simulated Price (10-day horizon)',
        yaxis_title='Frequency',
        barmode='overlay',
        plot_bgcolor='white', paper_bgcolor='white',
        height=420,
        margin=dict(l=50, r=40, t=80, b=50),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        xaxis=dict(showgrid=True, gridcolor='#F3F4F6'),
        yaxis=dict(showgrid=True, gridcolor='#F3F4F6'),
        annotations=[dict(
            text=msg, x=0.5, y=-0.12, xref='paper', yref='paper',
            showarrow=False, font=dict(size=9, color='gray'), align='center'
        )] if msg else [],
    )
    fig.show()

    print(f'Mean: ${mean_price:.2f} | Median: ${d["median_price"]:.2f}')
    print(f'90% range: ${lower_90:.2f} – ${upper_90:.2f}')
    print(f'Prob above threshold: {prob_above*100:.2f}%')

except ImportError:
    print('scipy not available — pip install scipy')
except Exception as e:
    print(f'Visualization skipped: {e}')


---
## 3. Option Chain Pressure — Max Pain & Dealer Gamma

Option pressure analysis identifies:
- **Max Pain**: the price where option sellers (typically dealers) face minimum payout — price often gravitates here near expiry
- **Dealer gamma exposure**: at what strikes dealers are long/short gamma, affecting their hedging behavior and creating price magnetism or repulsion zones

In [ ]:
pressure_data = client.get_option_pressure(SYMBOL)
print(json.dumps(pressure_data, indent=2))

In [ ]:
try:
    d = pressure_data

    # Price levels
    levels = {
        'Expiry Low':    d['expiry_low'],
        'Gamma Wall':    d['gamma_wall'],
        'Spot':          d['spot'],
        'Max Pain':      d['max_pain'],
        'Expected High': d['expected_high'],
        'Squeeze':       d['squeeze_price'],
        'Expiry High':   d['expiry_high'],
    }
    color_map = {
        'Expiry Low': '#94A3B8', 'Gamma Wall': '#F59E0B',
        'Spot': '#10B981', 'Max Pain': '#8B5CF6',
        'Expected High': '#3B82F6', 'Squeeze': '#EF4444',
        'Expiry High': '#94A3B8',
    }
    sorted_levels = sorted(levels.items(), key=lambda x: x[1])
    labels  = [l for l, _ in sorted_levels]
    prices  = [p for _, p in sorted_levels]
    colors  = [color_map[l] for l in labels]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Key Price Levels (Expiry {d["expiry"]})',
            'IV Breakdown'
        ),
        column_widths=[0.55, 0.45],
    )

    # Left: horizontal bar — price levels
    fig.add_trace(go.Bar(
        x=prices, y=labels,
        orientation='h',
        marker_color=colors,
        marker_line_width=0,
        opacity=0.8,
        text=[f'${p:.0f}' for p in prices],
        textposition='outside',
        name='',
    ), row=1, col=1)

    # Right: IV breakdown bars
    iv_labels = ['Call 25Δ', 'ATM', 'Put 25Δ']
    iv_vals   = [
        d['avg_call_iv'] * 100,
        d['avg_iv'] * 100,
        d['avg_put_iv'] * 100,
    ]
    iv_colors = ['#3B82F6', '#8B5CF6', '#EF4444']
    fig.add_trace(go.Bar(
        x=iv_labels, y=iv_vals,
        marker_color=iv_colors,
        marker_line_width=0,
        opacity=0.8,
        text=[f'{v:.1f}%' for v in iv_vals],
        textposition='outside',
        name='',
    ), row=1, col=2)

    fig.update_layout(
        title=dict(
            text=(
                f'Option Pressure — NVDA  |  Max Pain ${d["max_pain"]:.0f}  |  '
                f'Gamma Wall ${d["gamma_wall"]:.0f}  |  '
                f'Weekly EM ±${d["weekly_expected_move"]:.2f}  |  '
                f'Days to Expiry: {d["days_to_expiry"]}'
            ),
            font=dict(size=13)
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        height=420, showlegend=False,
        margin=dict(l=50, r=60, t=100, b=40),
        xaxis=dict(showgrid=True, gridcolor='#F3F4F6'),
        yaxis2=dict(showgrid=True, gridcolor='#F3F4F6'),
    )
    fig.update_xaxes(title_text='Price ($)', row=1, col=1)
    fig.update_yaxes(title_text='IV (%)', row=1, col=2)
    fig.show()

    print(f'Total OI: {d["total_oi"]:,} | Total Volume: {d["total_volume"]:,}')
    print(f'Gamma Wall Net: {d["gamma_wall_net"]:,.0f}')

except Exception as e:
    print(f'Visualization skipped: {e}')


---
## 4. Full Stock Analysis — Consensus Signal

`analyze_stock` aggregates all quant signals into a single JSON: momentum, volatility regime, IV percentile, option flow, and model consensus. Useful as a dashboard-level snapshot.

In [ ]:
analysis = client.analyze_stock(SYMBOL)
print(json.dumps(analysis, indent=2))

In [ ]:
try:
    d       = analysis
    signal  = d.get('signal', 'N/A')
    score   = d.get('confidence_score', 0)
    bullish = d.get('bullish_factors', [])
    bearish = d.get('bearish_factors', [])

    sig_color = {'Bullish': '#10B981', 'Bearish': '#EF4444'}.get(signal, '#F59E0B')

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=('Direction Score', 'Bullish Factors', 'Bearish Factors'),
        specs=[[{'type': 'indicator'}, {'type': 'table'}, {'type': 'table'}]],
        column_widths=[0.22, 0.39, 0.39],
    )

    # Gauge
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=score,
        title=dict(text=f'<b>{signal}</b>', font=dict(size=16, color=sig_color)),
        number=dict(suffix='/100', font=dict(size=20, color=sig_color)),
        gauge=dict(
            axis=dict(range=[0, 100], tickwidth=1, tickcolor='#888'),
            bar=dict(color=sig_color),
            bgcolor='white',
            steps=[
                dict(range=[0, 33],  color='#FEE2E2'),
                dict(range=[33, 66], color='#FEF3C7'),
                dict(range=[66, 100], color='#D1FAE5'),
            ],
        ),
    ), row=1, col=1)

    # Bullish table
    fig.add_trace(go.Table(
        header=dict(
            values=['<b>↑ Bullish</b>'],
            fill_color='#D1FAE5',
            font=dict(color='#065F46', size=12),
            align='left', height=30,
        ),
        cells=dict(
            values=[['\u2022 ' + f for f in bullish]],
            fill_color='white',
            font=dict(color='#1F2937', size=10),
            align='left', height=28,
        ),
    ), row=1, col=2)

    # Bearish table
    fig.add_trace(go.Table(
        header=dict(
            values=['<b>↓ Bearish</b>'],
            fill_color='#FEE2E2',
            font=dict(color='#991B1B', size=12),
            align='left', height=30,
        ),
        cells=dict(
            values=[['\u2022 ' + f for f in bearish]],
            fill_color='white',
            font=dict(color='#1F2937', size=10),
            align='left', height=28,
        ),
    ), row=1, col=3)

    fig.update_layout(
        title=dict(
            text=f'hpsilab — NVDA Quant Signal Dashboard  |  Spot ${d.get("spot", "")}',
            font=dict(size=14)
        ),
        plot_bgcolor='white', paper_bgcolor='white',
        height=380,
        margin=dict(l=20, r=20, t=80, b=20),
    )
    fig.show()

    # Tool coverage
    tool_responses = d.get('tool_responses', {})
    print('\nTool coverage:')
    for tool, resp in tool_responses.items():
        status = resp.get('status', 'unknown')
        icon = '\u2705' if status == 'ok' else '\U0001f512'
        print(f'  {icon}  {tool}: {status}')

except Exception as e:
    print(f'Visualization skipped: {e}')


---
## 5. (Pro) AI Direction Prediction

The AI prediction model combines:
- Regime detection (bull/bear/sideways via HMM)
- Regime-aware ensemble weighting
- Treasury yield curve features (DGS2 / DGS10 spread)
- VIX and relative strength signals
- VQC (Variational Quantum Circuit) layer via Qiskit

Returns a directional probability for the next trading day with a confidence-tiered signal.

> **Requires a Pro API key.** Get one at [hpsilab.com](https://hpsilab.com).

In [ ]:
# --- Option A: Kaggle Secrets (recommended) ---
# Add your key under Add-ons > Secrets with label HPSILAB_API_KEY

try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("HPSILAB_API_KEY")
    pro_client = HpsiMcpClient(api_key=api_key)
    prediction = pro_client.get_ai_prediction(SYMBOL)
    print(json.dumps(prediction, indent=2))
except Exception as e:
    print(f"Pro feature skipped: {e}")
    print("Add your API key to Kaggle Secrets as HPSILAB_API_KEY to unlock.")

---
## Summary

| Tool | Tier | What it gives you |
|------|------|-------------------|
| `get_iv_radar` | Free | IV term structure, skew, IV/HV ratio |
| `get_monte_carlo` | Free | 10-day price distribution, 90% confidence range |
| `get_option_pressure` | Free | Max Pain, dealer gamma exposure by strike |
| `analyze_stock` | Free | Aggregated quant consensus signal |
| `get_ai_prediction` | Pro | Regime-aware ML direction probability |

**Resources:**
- SDK: `pip install hpsilab-mcp`
- Docs & API: [hpsilab.com](https://hpsilab.com)
- MCP Server: `https://hpsilab.com/mcp` (Claude Desktop / any MCP client)
- GitHub: [github.com/haiyunsky/hpsilab-mcp-sdk](https://github.com/haiyunsky/hpsilab-mcp-sdk)

*These tools return research-oriented information and are not financial advice.*